In [1]:
# ===================================================================
# Autoformer + FAN  —  Multi-Seed × Multi-Horizon
# pred_len in [1, 3, 7, 30] days  |  seeds in [42, 123, 456, 789, 1011]
# کرنل میانگین متحرک به صورت پویا بر اساس pred_len تعیین می‌شود
# XAI snapshot فقط روی بهترین epoch (min val MAE) ذخیره می‌شود
# و هر بار که بهترین model آپدیت شود، snapshot هم آپدیت می‌شود
# ===================================================================

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
import os, time
import gc
import random

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

warnings.filterwarnings('ignore')


In [2]:
# -------------------------------------------------------------------
# 1. آماده‌سازی داده
# -------------------------------------------------------------------
df = pd.read_csv('ilam_weather.csv', index_col=0, parse_dates=True)

def create_advanced_features(dataframe, col_target='tmin'):
    df = dataframe.copy()
    for lag in [1, 2, 3, 7, 14, 30]:
        df[f'{col_target}_lag{lag}'] = df[col_target].shift(lag)
    for window in [7, 14, 30]:
        df[f'{col_target}_rolling_mean_{window}'] = df[col_target].shift(1).rolling(window).mean()
        df[f'{col_target}_rolling_std_{window}']  = df[col_target].shift(1).rolling(window).std()
        df[f'{col_target}_rolling_min_{window}']  = df[col_target].shift(1).rolling(window).min()
        df[f'{col_target}_rolling_max_{window}']  = df[col_target].shift(1).rolling(window).max()
    df[f'{col_target}_ewm_7']  = df[col_target].shift(1).ewm(span=7).mean()
    df[f'{col_target}_ewm_30'] = df[col_target].shift(1).ewm(span=30).mean()
    df[f'{col_target}_diff_1'] = df[col_target].shift(1).diff(1)
    df[f'{col_target}_diff_7'] = df[col_target].shift(1).diff(7)
    df['temp_range']     = df['tmax'].shift(1) - df['tmin'].shift(1)
    df['humidity_range'] = df['umax'].shift(1) - df['umin'].shift(1)
    df['pressure_range'] = df['p0max'].shift(1) - df['p0min'].shift(1)
    df['tmax']  = df['tmax'].shift(1)
    df['p0max'] = df['p0max'].shift(1)
    df['p0min'] = df['p0min'].shift(1)
    df['sshn']  = df['sshn'].shift(1)
    df['umin']  = df['umin'].shift(1)
    df['umax']  = df['umax'].shift(1)
    df['day_of_year'] = df.index.dayofyear
    df['month']       = df.index.month
    df['day_of_week'] = df.index.dayofweek
    df['month_sin']       = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']       = np.cos(2 * np.pi * df['month'] / 12)
    df['day_of_year_sin'] = np.sin(2 * np.pi * df['day_of_year'] / 365.25)
    df['day_of_year_cos'] = np.cos(2 * np.pi * df['day_of_year'] / 365.25)
    df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
    df.drop(['day_of_year', 'month', 'day_of_week'], axis=1, inplace=True)
    df.dropna(inplace=True)
    return df

df = create_advanced_features(df)
target_idx = df.columns.get_loc('tmin')

cyclical_features = ['month_sin', 'month_cos', 'day_of_year_sin',
                     'day_of_year_cos', 'day_of_week_sin', 'day_of_week_cos']
num_cyclical = len(cyclical_features)
cols  = list(df.columns)
data  = df[cols].values

train_size = int(0.7 * len(data))
val_size   = int(0.15 * len(data))
train_data = data[:train_size]
val_data   = data[train_size:train_size + val_size]
test_data  = data[train_size + val_size:]

print(f'Train: {train_data.shape}, Val: {val_data.shape}, Test: {test_data.shape}')
print(f'Number of features: {len(cols)} | Target index: {target_idx}')


Train: (5091, 38), Val: (1091, 38), Test: (1092, 38)
Number of features: 38 | Target index: 0


In [3]:
# -------------------------------------------------------------------
# 2. پارامترهای ثابت
# -------------------------------------------------------------------
SEQLEN        = 365
LABELLEN      = 182
BATCH_SIZE    = 64
PRED_HORIZONS = [1, 3, 7, 30]


In [4]:
# -------------------------------------------------------------------
# 3. Dataset کشویی
# -------------------------------------------------------------------
class SlidingWindowDataset(Dataset):
    def __init__(self, data, target_col_idx, seq_len=365, label_len=182, pred_len=1):
        self.data           = data
        self.target_col_idx = target_col_idx
        self.seq_len        = seq_len
        self.label_len      = label_len
        self.pred_len       = pred_len
        sequences, dec_inputs, targets = [], [], []
        for i in range(len(data) - seq_len - pred_len + 1):
            seq           = data[i:i + seq_len]
            dec_start_idx = i + seq_len - label_len
            label_part    = data[dec_start_idx:i + seq_len]
            label_mean    = np.mean(label_part, axis=0, keepdims=True)
            pred_part     = np.repeat(label_mean, pred_len, axis=0)
            dec_inp       = np.concatenate([label_part, pred_part], axis=0)
            target        = data[i + seq_len:i + seq_len + pred_len, target_col_idx]
            sequences.append(seq)
            dec_inputs.append(dec_inp)
            targets.append(target)
        self.sequences  = np.array(sequences,  dtype=np.float32)
        self.dec_inputs = np.array(dec_inputs, dtype=np.float32)
        self.targets    = np.array(targets,    dtype=np.float32)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return (torch.FloatTensor(self.sequences[idx]),
                torch.FloatTensor(self.dec_inputs[idx]),
                torch.FloatTensor(self.targets[idx]))


In [5]:
# -------------------------------------------------------------------
# 4. معماری Autoformer + FAN
# -------------------------------------------------------------------
class FANLayer(nn.Module):
    def __init__(self, num_features, num_cyclical_features=0, K=4):
        super().__init__()
        self.K = K
        self.num_features          = num_features
        self.num_cyclical_features = num_cyclical_features
        self.num_normal_features   = num_features - num_cyclical_features
        self.denorm_mlp = nn.Sequential(
            nn.Linear(self.num_normal_features * 4, 256),
            nn.GELU(), nn.Dropout(0.1),
            nn.Linear(256, self.num_normal_features)
        )
        self.saved_nonstat_nc = None
        self.saved_input_enc  = None

    def forward(self, x, mode='norm', context_encoder=False,
                original_x=None, target_idx_in_normal=None):
        if self.num_cyclical_features > 0:
            x_normal   = x[..., :-self.num_cyclical_features]
            x_cyclical = x[..., -self.num_cyclical_features:]
        else:
            x_normal, x_cyclical = x, None
        if mode == 'norm':
            X_fft          = torch.fft.rfft(x_normal, dim=1)
            Amp_mean       = torch.abs(X_fft).mean(dim=(0, 2))
            _, topk_idx    = torch.topk(Amp_mean, k=min(self.K, len(Amp_mean)))
            X_fft_filtered = torch.zeros_like(X_fft)
            X_fft_filtered[:, topk_idx, :] = X_fft[:, topk_idx, :]
            X_nonstat = torch.fft.irfft(X_fft_filtered, n=x_normal.shape[1], dim=1)
            X_res     = x_normal - X_nonstat
            if context_encoder:
                self.saved_nonstat_nc = X_nonstat.detach()
                self.saved_input_enc  = x_normal.detach()
            return torch.cat([X_res, x_cyclical], dim=-1) if x_cyclical is not None else X_res
        elif mode == 'denorm':
            nonstat_last = self.saved_nonstat_nc[:, -1, :]
            nonstat_mean = self.saved_nonstat_nc.mean(dim=1)
            input_last   = self.saved_input_enc[:, -1, :]
            input_mean   = self.saved_input_enc.mean(dim=1)
            mlp_in       = torch.cat([nonstat_last, nonstat_mean, input_last, input_mean], dim=-1)
            Y_pred       = self.denorm_mlp(mlp_in)
            pred_len     = x.shape[1]
            Y_pred       = Y_pred.unsqueeze(1).repeat(1, pred_len, 1)
            if target_idx_in_normal is not None:
                t_ns = Y_pred[:, :, target_idx_in_normal:target_idx_in_normal + 1]
            else:
                t_ns = Y_pred.mean(dim=-1, keepdim=True)
            return x + t_ns


class SimpleDecompositionBlock(nn.Module):
    def __init__(self, kernel_size=25):
        super().__init__()
        self.moving_avg = nn.AvgPool1d(kernel_size=kernel_size, stride=1, padding=kernel_size // 2)

    def forward(self, x):
        trend    = self.moving_avg(x.permute(0, 2, 1)).permute(0, 2, 1)
        seasonal = x - trend
        return seasonal, trend


class AutoCorrelation(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.1, factor=1):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_keys  = d_model // n_heads
        self.dropout = nn.Dropout(dropout)
        self.factor  = factor

    def time_delay_agg(self, Q, K, V, topk):
        B, LQ, H, D = Q.shape
        _, LK, _, _ = K.shape
        L_max = max(LQ, LK)
        if LQ < L_max: Q = F.pad(Q, (0,0,0,0,0,L_max-LQ))
        if LK < L_max:
            K = F.pad(K, (0,0,0,0,0,L_max-LK))
            V = F.pad(V, (0,0,0,0,0,L_max-LK))
        corr      = torch.fft.irfft(
            torch.fft.rfft(Q, dim=1) * torch.conj(torch.fft.rfft(K, dim=1)),
            n=L_max, dim=1)
        corr_mean = corr.mean(dim=(0, 2, 3)) / np.sqrt(self.d_model)
        _, tidx   = torch.topk(corr_mean.abs(), k=topk)
        weights   = F.softmax(corr_mean[tidx], dim=0)
        out       = torch.zeros_like(Q)
        for i in range(topk):
            out = out + torch.roll(V, shifts=int(tidx[i].item()), dims=1) * weights[i].view(1,1,1,1)
        return out[:, :LQ, :, :]

    def forward(self, Q, K, V):
        B, L, _ = Q.shape
        _, S, _ = K.shape
        H = self.n_heads
        Q = Q.view(B, L, H, self.d_keys)
        K = K.view(B, S, H, self.d_keys)
        V = V.view(B, S, H, self.d_keys)
        out = self.time_delay_agg(Q, K, V, topk=max(1, int(self.factor * np.log(L))))
        return self.dropout(out.reshape(B, L, -1))


class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1, kernel_size=25):
        super().__init__()
        self.autocorrelation = AutoCorrelation(d_model, n_heads, dropout)
        self.decomp1 = SimpleDecompositionBlock(kernel_size=kernel_size)
        self.decomp2 = SimpleDecompositionBlock(kernel_size=kernel_size)
        self.ff      = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        x    = self.norm1(x + self.autocorrelation(x, x, x))
        s, _ = self.decomp2(x + self.ff(self.decomp1(x)[0]))
        return s


class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1, kernel_size=25):
        super().__init__()
        self.self_corr  = AutoCorrelation(d_model, n_heads, dropout)
        self.cross_corr = AutoCorrelation(d_model, n_heads, dropout)
        self.decomp1    = SimpleDecompositionBlock(kernel_size=kernel_size)
        self.decomp2    = SimpleDecompositionBlock(kernel_size=kernel_size)
        self.decomp3    = SimpleDecompositionBlock(kernel_size=kernel_size)
        self.ff         = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ff, d_model), nn.Dropout(dropout))
        self.norm1      = nn.LayerNorm(d_model)
        self.norm2      = nn.LayerNorm(d_model)
        self.norm3      = nn.LayerNorm(d_model)
        self.trend_proj = nn.Linear(d_model, d_model)

    def forward(self, x, enc_out):
        x     = self.norm1(x + self.self_corr(x, x, x))
        s1, _ = self.decomp1(x)
        s1    = self.norm2(s1 + self.cross_corr(s1, enc_out, enc_out))
        s2, _ = self.decomp2(s1)
        s2    = self.norm3(s2 + self.ff(s2))
        s3, _ = self.decomp3(s2)
        trend = s1 + self.trend_proj(s2) + self.trend_proj(s3)
        return s3, trend


class Autoformer(nn.Module):
    def __init__(self, n_features, d_model, n_heads, n_encoder_layers, n_decoder_layers,
                 d_ff, seq_len, pred_len, dropout,
                 num_cyclical_features=0, fan_K=4, target_idx=None,
                 moving_avg_kernel=25):
        super().__init__()
        self.pred_len = pred_len
        self.target_idx_in_normal = (
            target_idx
            if target_idx is not None and target_idx < n_features - num_cyclical_features
            else None
        )
        self.fan_layer     = FANLayer(n_features, num_cyclical_features, K=fan_K)
        self.enc_embedding = nn.Linear(n_features, d_model)
        self.dec_embedding = nn.Linear(n_features, d_model)
        self.pos_enc = nn.Parameter(torch.randn(1, seq_len, d_model) * 0.02)
        self.pos_dec = nn.Parameter(torch.randn(1, seq_len + pred_len, d_model) * 0.02)
        self.encoder_layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout, kernel_size=moving_avg_kernel)
            for _ in range(n_encoder_layers)])
        self.decoder_layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout, kernel_size=moving_avg_kernel)
            for _ in range(n_decoder_layers)])
        self.projection = nn.Linear(d_model, 1)

    def forward(self, x_enc, x_dec):
        enc_out = self.enc_embedding(
            self.fan_layer(x_enc, mode='norm', context_encoder=True)
        ) + self.pos_enc[:, :x_enc.size(1), :]
        for layer in self.encoder_layers:
            enc_out = layer(enc_out)
        dec_out = self.dec_embedding(
            self.fan_layer(x_dec, mode='norm', context_encoder=False)
        ) + self.pos_dec[:, :x_dec.size(1), :]
        seasonal_part = trend_part = 0
        for layer in self.decoder_layers:
            seasonal_part, trend_part = layer(dec_out, enc_out)
        dec_out = dec_out + seasonal_part + trend_part
        output  = self.projection(dec_out[:, -self.pred_len:, :])
        return self.fan_layer(
            output, mode='denorm', context_encoder=True,
            original_x=x_enc,
            target_idx_in_normal=self.target_idx_in_normal
        ).squeeze(-1)


In [6]:
# -------------------------------------------------------------------
# 5. XAI: LayerWiseInterpreter و AdvancedRealtimeVisualizer
# -------------------------------------------------------------------
class LayerWiseInterpreter:
    def __init__(self, model, feature_names, device='cpu'):
        self.model            = model
        self.feature_names    = feature_names
        self.device           = device
        self.layer_activations = {}
        self.layer_gradients   = {}
        self.input_data        = None
        self.register_hooks()

    def register_hooks(self):
        for idx, layer in enumerate(self.model.encoder_layers):
            layer.autocorrelation.register_forward_hook(self.create_activation_hook(f'encoder_attn_{idx}'))
            layer.autocorrelation.register_full_backward_hook(self.create_gradient_hook(f'encoder_attn_{idx}'))
            layer.ff.register_forward_hook(self.create_activation_hook(f'encoder_ff_{idx}'))
            layer.ff.register_full_backward_hook(self.create_gradient_hook(f'encoder_ff_{idx}'))
        for idx, layer in enumerate(self.model.decoder_layers):
            layer.self_corr.register_forward_hook(self.create_activation_hook(f'decoder_self_attn_{idx}'))
            layer.self_corr.register_full_backward_hook(self.create_gradient_hook(f'decoder_self_attn_{idx}'))
            layer.cross_corr.register_forward_hook(self.create_activation_hook(f'decoder_cross_attn_{idx}'))
            layer.cross_corr.register_full_backward_hook(self.create_gradient_hook(f'decoder_cross_attn_{idx}'))
            layer.ff.register_forward_hook(self.create_activation_hook(f'decoder_ff_{idx}'))
            layer.ff.register_full_backward_hook(self.create_gradient_hook(f'decoder_ff_{idx}'))

    def create_activation_hook(self, name):
        def hook(module, input, output):
            self.layer_activations[name] = output.detach()
        return hook

    def create_gradient_hook(self, name):
        def hook(module, grad_input, grad_output):
            if grad_output[0] is not None:
                self.layer_gradients[name] = grad_output[0].detach()
        return hook

    def store_input(self, x_enc):
        self.input_data = x_enc.detach()

    def compute_feature_importance(self):
        if not self.layer_activations or not self.layer_gradients:
            return {}
        layer_importance = {}
        for layer_name in sorted(self.layer_activations.keys()):
            if layer_name not in self.layer_gradients:
                continue
            activation     = self.layer_activations[layer_name]
            gradient       = self.layer_gradients[layer_name]
            importance_map = (gradient.abs() * activation.abs()).mean(dim=(0, 1))
            if importance_map.max() > 0:
                importance_map = importance_map / importance_map.max()
            importance_np = importance_map.cpu().numpy()
            if len(importance_np) < len(self.feature_names):
                repeat_factor = int(np.ceil(len(self.feature_names) / len(importance_np)))
                importance_np = np.tile(importance_np, repeat_factor)[:len(self.feature_names)]
            elif len(importance_np) > len(self.feature_names):
                importance_np = importance_np[:len(self.feature_names)]
            if len(importance_np) == len(self.feature_names):
                top_k       = min(5, len(self.feature_names))
                top_indices = np.argsort(importance_np)[-top_k:][::-1]
                layer_importance[layer_name] = {
                    'importance':  importance_np,
                    'top_features': [(self.feature_names[i], float(importance_np[i])) for i in top_indices]
                }
        return layer_importance

    def compute_input_gradient_importance(self):
        try:
            if self.input_data is None:
                return None
            all_gradients = []
            for gradient in self.layer_gradients.values():
                grad_mean = gradient.abs().mean(dim=1).squeeze(0)
                all_gradients.append(grad_mean)
            if all_gradients:
                avg_gradient = torch.stack(all_gradients).mean(dim=0).cpu().numpy()
                if len(avg_gradient) < len(self.feature_names):
                    repeat_factor = max(1, len(self.feature_names) // len(avg_gradient))
                    return np.tile(avg_gradient, repeat_factor)[:len(self.feature_names)]
                elif len(avg_gradient) == len(self.feature_names):
                    return avg_gradient
        except Exception as e:
            print(f'Warning in compute_input_gradient_importance: {e}')
        return None

    def clear_cache(self):
        self.layer_activations.clear()
        self.layer_gradients.clear()
        self.input_data = None


class AdvancedRealtimeVisualizer:
    def __init__(self, feature_names, save_dir=None):
        self.feature_names = feature_names
        self.history       = []
        self.save_dir      = save_dir

    def add_snapshot(self, epoch, layer_importance, input_importance=None, val_mae=None):
        if not layer_importance:
            return
        self.history.append({
            'epoch': epoch,
            'importance': layer_importance,
            'input_importance': input_importance,
            'val_mae': val_mae,
        })

    def visualize_all_layers_evolution(self, top_k=10):
        if not self.history:
            return None
        all_importances = {}
        for snapshot in self.history:
            for layer_name, imp_data in snapshot['importance'].items():
                if layer_name not in all_importances:
                    all_importances[layer_name] = {}
                for feat_name, imp_score in imp_data['top_features']:
                    if feat_name not in all_importances[layer_name]:
                        all_importances[layer_name][feat_name] = []
                    all_importances[layer_name][feat_name].append(imp_score)
        layer_names = sorted(all_importances.keys())
        n_layers    = len(layer_names)
        if n_layers == 0:
            return None
        vertical_spacing = min(0.05, (1.0 / (n_layers - 1)) * 0.8) if n_layers > 1 else 0.05
        fig = make_subplots(rows=n_layers, cols=1,
                            subplot_titles=[f'Layer: {name}' for name in layer_names],
                            vertical_spacing=vertical_spacing)
        colors = ['#FF6B6B','#4ECDC4','#45B7D1','#FFA07A','#98D8C8',
                  '#F7DC6F','#BB8FCE','#85C1E2','#F8B88B','#99CC99']
        for layer_idx, layer_name in enumerate(layer_names):
            layer_data     = all_importances[layer_name]
            avg_importance = {feat: np.mean(scores) for feat, scores in layer_data.items()}
            top_feats      = sorted(avg_importance.items(), key=lambda x: x[1], reverse=True)[:top_k]
            for feat_idx, (feat_name, _) in enumerate(top_feats):
                if feat_name in layer_data:
                    epochs = list(range(len(layer_data[feat_name])))
                    fig.add_trace(go.Scatter(
                        x=epochs, y=layer_data[feat_name],
                        mode='lines+markers', name=feat_name,
                        line=dict(color=colors[feat_idx % len(colors)], width=2),
                        marker=dict(size=4),
                        showlegend=(layer_idx == 0)),
                        row=layer_idx + 1, col=1)
            fig.update_xaxes(title_text='Snapshot', row=layer_idx + 1, col=1)
            fig.update_yaxes(title_text='Importance', row=layer_idx + 1, col=1)
        total_height = max(200, min(300, 3000 // n_layers)) * n_layers
        fig.update_layout(
            title='Feature Importance Evolution — All Layers',
            font=dict(family='Arial', size=13),
            height=total_height, template='plotly_white',
            hovermode='x unified', showlegend=True)
        return fig

    def visualize_layer_importance_comparison(self):
        if not self.history:
            return None
        last_snapshot  = self.history[-1]
        layer_importance = last_snapshot['importance']
        layer_names      = sorted(layer_importance.keys())
        if not layer_names:
            return None
        max_layers_per_fig = 6
        if len(layer_names) <= max_layers_per_fig:
            fig = make_subplots(
                rows=1, cols=len(layer_names),
                subplot_titles=[f'{name}' for name in layer_names],
                specs=[[{'type': 'bar'} for _ in range(len(layer_names))]],
                horizontal_spacing=0.05)
            for col_idx, layer_name in enumerate(layer_names):
                imp_data     = layer_importance[layer_name]
                top_features = [f[0] for f in imp_data['top_features']]
                top_values   = [f[1] for f in imp_data['top_features']]
                fig.add_trace(go.Bar(
                    y=top_features, x=top_values, orientation='h',
                    name=layer_name, showlegend=False,
                    marker=dict(color=top_values, colorscale='Viridis', showscale=(col_idx == 0)),
                    text=[f'{v:.3f}' for v in top_values], textposition='auto'),
                    row=1, col=col_idx + 1)
                fig.update_xaxes(title_text='Score', row=1, col=col_idx + 1)
            fig.update_layout(
                title='Feature Importance — Layer Comparison',
                font=dict(family='Arial', size=13),
                height=500, template='plotly_white', hovermode='y')
        else:
            fig = go.Figure()
            for layer_name in layer_names:
                imp_data     = layer_importance[layer_name]
                top_features = [f[0] for f in imp_data['top_features']]
                top_values   = [f[1] for f in imp_data['top_features']]
                fig.add_trace(go.Bar(
                    y=top_features, x=top_values, orientation='h',
                    name=layer_name,
                    text=[f'{v:.3f}' for v in top_values], textposition='auto'))
            fig.update_layout(
                title='Feature Importance — All Layers',
                font=dict(family='Arial', size=13),
                xaxis_title='Importance Score', yaxis_title='Features',
                height=600, template='plotly_white',
                barmode='group', showlegend=True)
        return fig

    def visualize_attention_heatmap(self):
        if not self.history:
            return None
        last_snapshot    = self.history[-1]
        layer_importance = last_snapshot['importance']
        encoder_layers   = {k: v for k, v in layer_importance.items() if 'encoder' in k}
        decoder_layers   = {k: v for k, v in layer_importance.items() if 'decoder' in k}
        if not encoder_layers and not decoder_layers:
            return None
        n_cols = (1 if encoder_layers else 0) + (1 if decoder_layers else 0)
        titles = []
        if encoder_layers: titles.append('Encoder Layers')
        if decoder_layers: titles.append('Decoder Layers')
        fig = make_subplots(
            rows=1, cols=n_cols,
            subplot_titles=titles,
            specs=[[{'type': 'heatmap'} for _ in range(n_cols)]])
        col_idx = 1
        for layers_dict, label in [(encoder_layers, 'Encoder'), (decoder_layers, 'Decoder')]:
            if not layers_dict:
                continue
            matrix      = []
            layer_names = sorted(layers_dict.keys())
            for layer in layer_names:
                row_data = []
                for feat in self.feature_names[:10]:
                    val = 0
                    for feat_name, imp_score in layers_dict[layer]['top_features']:
                        if feat_name == feat:
                            val = imp_score
                            break
                    row_data.append(val)
                matrix.append(row_data)
            fig.add_trace(go.Heatmap(
                z=matrix, y=layer_names, x=self.feature_names[:10],
                colorscale='RdYlBu_r', showscale=(col_idx == 1),
                name=label),
                row=1, col=col_idx)
            col_idx += 1
        fig.update_layout(
            title='Attention Heatmap — Layer Importance',
            font=dict(family='Arial', size=13),
            height=max(400, (len(encoder_layers) + len(decoder_layers)) * 50 + 100),
            template='plotly_white')
        return fig

    def visualize_temporal_importance_trend(self):
        if not self.history or len(self.history) < 2:
            return None
        feature_trends = {feat: [] for feat in self.feature_names}
        epochs = []
        for snapshot in self.history:
            epochs.append(snapshot['epoch'])
            for feat in self.feature_names:
                feat_importance = []
                for layer_name, imp_data in snapshot['importance'].items():
                    for feat_name, imp_score in imp_data['top_features']:
                        if feat_name == feat:
                            feat_importance.append(imp_score)
                feature_trends[feat].append(np.mean(feat_importance) if feat_importance else 0)
        top_features_overall = sorted(
            [(feat, np.mean(scores)) for feat, scores in feature_trends.items()],
            key=lambda x: x[1], reverse=True)[:10]
        fig    = go.Figure()
        colors = ['#FF6B6B','#4ECDC4','#45B7D1','#FFA07A','#98D8C8',
                  '#F7DC6F','#BB8FCE','#85C1E2','#F8B88B','#99CC99']
        for idx, (feat, _) in enumerate(top_features_overall):
            fig.add_trace(go.Scatter(
                x=epochs, y=feature_trends[feat],
                mode='lines+markers', name=feat,
                line=dict(color=colors[idx], width=3),
                marker=dict(size=6)))
        fig.update_layout(
            title='Temporal Trend — Feature Importance Over Epochs',
            font=dict(family='Arial', size=13),
            xaxis_title='Epoch', yaxis_title='Avg Importance Score',
            height=600, template='plotly_white',
            hovermode='x unified', showlegend=True)
        return fig


In [7]:
# -------------------------------------------------------------------
# 6. توابع ارزیابی
# -------------------------------------------------------------------
def calculate_smape(y_true, y_pred):
    return np.mean(
        np.abs(y_true - y_pred) / ((np.abs(y_true) + np.abs(y_pred)) / 2 + 1e-8)
    ) * 100


def evaluate_model(model, dataloader, device, criterion=None):
    model.eval()
    all_preds, all_targets, total_loss = [], [], 0.0
    with torch.no_grad():
        for x_enc, x_dec, y in dataloader:
            x_enc, x_dec, y = x_enc.to(device), x_dec.to(device), y.to(device)
            out = model(x_enc, x_dec)
            if criterion:
                total_loss += criterion(out, y).item()
            all_preds.append(out.cpu().numpy())
            all_targets.append(y.cpu().numpy())
    preds   = np.concatenate(all_preds).flatten()
    targets = np.concatenate(all_targets).flatten()
    return {
        'mae':           mean_absolute_error(targets, preds),
        'rmse':          np.sqrt(mean_squared_error(targets, preds)),
        'r2_score':      r2_score(targets, preds),
        'smape_percent': calculate_smape(targets, preds),
        'loss':          total_loss / len(dataloader) if criterion else None,
        'preds':   preds,
        'targets': targets,
    }


In [8]:
# -------------------------------------------------------------------
# 7. تابع ذخیره snapshot XAI برای بهترین epoch
# -------------------------------------------------------------------
def save_best_epoch_xai_snapshot(visualizer, save_dir, seed, pred_len, best_epoch, best_val_mae):
    """
    تمام نمودارهای XAI را فقط برای بهترین epoch محاسبه و ذخیره می‌کند.
    هر بار که بهترین model آپدیت شود، این تابع فراخوانی شده و
    فایل‌های قبلی با نسخه جدید جایگزین می‌شوند.
    """
    os.makedirs(save_dir, exist_ok=True)

    charts = [
        ('attention_heatmap_best',      visualizer.visualize_attention_heatmap),
        ('layer_importance_best',        visualizer.visualize_layer_importance_comparison),
        ('temporal_trend_best',          visualizer.visualize_temporal_importance_trend),
        ('all_layers_evolution_best',    visualizer.visualize_all_layers_evolution),
    ]

    for name, method in charts:
        try:
            fig = method()
            if fig is not None:
                fig.update_layout(
                    title=fig.layout.title.text +
                    f'  |  Best Epoch={best_epoch+1}  Val MAE={best_val_mae:.4f}'
                    f'  |  seed={seed}  pred_len={pred_len}')
                fig.write_html(f'{save_dir}/{name}.html')
        except Exception as e:
            print(f'  Warning: could not save {name}: {e}')

    # نمودار پیشرفت val MAE
    val_maes = [s['val_mae'] for s in visualizer.history if s.get('val_mae') is not None]
    if val_maes:
        epochs_list = [s['epoch'] for s in visualizer.history if s.get('val_mae') is not None]
        fig_progress = go.Figure()
        fig_progress.add_trace(go.Scatter(
            x=epochs_list, y=val_maes,
            mode='lines+markers', name='Val MAE',
            line=dict(color='royalblue', width=2), marker=dict(size=5)))
        fig_progress.add_vline(
            x=best_epoch, line_dash='dash', line_color='red', opacity=0.7,
            annotation_text=f'Best Epoch={best_epoch+1}  MAE={best_val_mae:.4f}',
            annotation_position='top left')
        fig_progress.update_layout(
            title=f'Training Progress — Best Epoch={best_epoch+1} | seed={seed} pred_len={pred_len}',
            xaxis_title='Epoch', yaxis_title='Validation MAE',
            template='plotly_white', font=dict(family='Arial', size=13), height=400)
        fig_progress.write_html(f'{save_dir}/training_progress_best.html')

    print(f'  ✅ XAI snapshot آپدیت شد — Best Epoch={best_epoch+1} | Val MAE={best_val_mae:.4f}')


In [9]:
# -------------------------------------------------------------------
# 8. تابع آموزش با XAI روی بهترین epoch
# -------------------------------------------------------------------
def train_with_best_epoch_xai(model, train_loader, val_loader, feature_names,
                               epochs=50, lr=5e-4, device='cpu',
                               seed=None, pred_len=1, verbose=False):
    """
    - در هر epoch یک snapshot از XAI گرفته می‌شود.
    - هر بار که val MAE بهبود یابد (best model آپدیت شود)،
      تمام نمودارهای XAI با snapshot آن epoch جایگزین می‌شوند.
    - در پایان، نمودارهای نهایی متعلق به بهترین epoch هستند.
    """
    save_dir = f'xai_outputs/seed{seed}_pred{pred_len}'
    os.makedirs(save_dir, exist_ok=True)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, epochs=epochs,
        steps_per_epoch=len(train_loader),
        pct_start=0.1, anneal_strategy='cos')
    criterion = nn.MSELoss()

    interpreter = LayerWiseInterpreter(model, feature_names=feature_names, device=device)
    visualizer  = AdvancedRealtimeVisualizer(feature_names=feature_names, save_dir=save_dir)

    best_val_mae  = float('inf')
    best_epoch    = 0
    patience_ctr  = 0
    patience      = 20

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for batch_idx, (x_enc, x_dec, y) in enumerate(train_loader):
            x_enc, x_dec, y = x_enc.to(device), x_dec.to(device), y.to(device)
            interpreter.store_input(x_enc)
            optimizer.zero_grad()
            out  = model(x_enc, x_dec)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        # گرفتن XAI snapshot از آخرین batch هر epoch
        interpreter.clear_cache()
        model.train()
        last_x_enc, last_x_dec, last_y = x_enc, x_dec, y
        out_interp  = model(last_x_enc, last_x_dec)
        loss_interp = criterion(out_interp, last_y)
        loss_interp.backward()
        importance_dict   = interpreter.compute_feature_importance()
        input_importance  = interpreter.compute_input_gradient_importance()

        val_res   = evaluate_model(model, val_loader, device, criterion)
        val_mae   = val_res['mae']
        val_smape = val_res['smape_percent']

        # snapshot را به تاریخچه visualizer اضافه کن
        if importance_dict:
            visualizer.add_snapshot(epoch, importance_dict, input_importance, val_mae)

        if verbose and (epoch % 5 == 0 or epoch == epochs - 1):
            print(f'  Epoch {epoch+1:2d} | Train MSE: {total_loss/len(train_loader):.4f} '
                  f'| Val MAE: {val_mae:.4f} | Val sMAPE: {val_smape:.4f}')

        # اگر بهترین model آپدیت شد → snapshot XAI را هم آپدیت کن
        if val_mae < best_val_mae:
            best_val_mae = val_mae
            best_epoch   = epoch
            patience_ctr = 0
            torch.save({'model_state_dict': model.state_dict(), 'val_mae': best_val_mae},
                       f'best_autoformer_seed{seed}_pred{pred_len}.pth')
            # آپدیت فوری همه نمودارهای XAI برای این epoch
            if importance_dict:
                save_best_epoch_xai_snapshot(
                    visualizer, save_dir, seed, pred_len, best_epoch, best_val_mae)
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                if verbose:
                    print(f'  Early stopping at epoch {epoch+1} — Best epoch was {best_epoch+1}')
                break

    print(f'  🏆 بهترین epoch: {best_epoch+1} | Val MAE: {best_val_mae:.4f}')
    return model


In [10]:
# -------------------------------------------------------------------
# 9. حلقه اصلی
# -------------------------------------------------------------------
if __name__ == '__main__':
    seeds  = [42, 123, 456, 789, 1011]
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Device: {device}')

    all_results = []

    for pred_len in PRED_HORIZONS:
        print(f"\n{'#'*60}")
        print(f'#  افق پیش‌بینی: {pred_len} روز')
        print(f"{'#'*60}")

        kernel_size = pred_len + 1 if pred_len % 2 == 0 else pred_len + 2
        print(f'کرنل پویا برای pred_len={pred_len} → kernel_size={kernel_size}')

        train_ds = SlidingWindowDataset(train_data, target_idx, SEQLEN, LABELLEN, pred_len)
        val_ds   = SlidingWindowDataset(val_data,   target_idx, SEQLEN, LABELLEN, pred_len)
        test_ds  = SlidingWindowDataset(test_data,  target_idx, SEQLEN, LABELLEN, pred_len)

        horizon_results = []

        for seed in seeds:
            print(f"\n{'='*60}")
            print(f'seed={seed} | pred_len={pred_len} | kernel={kernel_size}')

            random.seed(seed)
            np.random.seed(seed)
            torch.manual_seed(seed)
            if torch.cuda.is_available():
                torch.cuda.manual_seed(seed)
                torch.cuda.manual_seed_all(seed)
                torch.backends.cudnn.deterministic = True
                torch.backends.cudnn.benchmark     = False
            os.environ['PYTHONHASHSEED'] = str(seed)
            torch.cuda.empty_cache()
            gc.collect()

            model = Autoformer(
                n_features=len(df.columns), d_model=64, n_heads=4,
                n_encoder_layers=4, n_decoder_layers=2, d_ff=256,
                seq_len=SEQLEN, pred_len=pred_len,
                dropout=0.15, num_cyclical_features=num_cyclical,
                fan_K=4, target_idx=target_idx,
                moving_avg_kernel=kernel_size
            ).to(device)
            print(f'Total params: {sum(p.numel() for p in model.parameters()):,}')

            train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True,  drop_last=True)
            val_loader   = DataLoader(val_ds,   BATCH_SIZE, shuffle=False)
            test_loader  = DataLoader(test_ds,  BATCH_SIZE, shuffle=False)

            t0 = time.perf_counter()
            model = train_with_best_epoch_xai(
                model, train_loader, val_loader,
                feature_names=list(df.columns),
                epochs=50, lr=5e-4, device=device,
                seed=seed, pred_len=pred_len, verbose=True)
            print(f'زمان آموزش: {time.perf_counter()-t0:.1f} ثانیه')

            ckpt = torch.load(f'best_autoformer_seed{seed}_pred{pred_len}.pth',
                              map_location=device, weights_only=False)
            model.load_state_dict(ckpt['model_state_dict'])

            res = evaluate_model(model, test_loader, device)
            print(f"نتایج تست: MAE={res['mae']:.2f}°C | RMSE={res['rmse']:.2f}°C | "
                  f"R²={res['r2_score']:.4f} | sMAPE={res['smape_percent']:.2f}%")

            row = {
                'pred_len':      pred_len,
                'seed':          seed,
                'kernel_size':   kernel_size,
                'mae':           res['mae'],
                'rmse':          res['rmse'],
                'r2_score':      res['r2_score'],
                'smape_percent': res['smape_percent'],
            }
            horizon_results.append(row)
            all_results.append(row)

            del model
            torch.cuda.empty_cache()
            gc.collect()

        hr_df = pd.DataFrame(horizon_results)
        print(f"\n📊 خلاصه افق {pred_len} روز (میانگین ± std روی {len(seeds)} seed):")
        for metric in ['mae', 'rmse', 'r2_score', 'smape_percent']:
            print(f'  {metric:15s}: {hr_df[metric].mean():.4f} ± {hr_df[metric].std():.4f}')

    # -------------------------------------------------------------------
    # 10. ذخیره نتایج نهایی
    # -------------------------------------------------------------------
    results_df = pd.DataFrame(all_results)
    results_df.to_csv('fanformer_auto_best_epoch_results.csv', index=False)
    print('\n✅ نتایج در fanformer_auto_best_epoch_results.csv ذخیره شد.')

    summary = results_df.groupby('pred_len')[['mae','rmse','r2_score','smape_percent']].agg(['mean','std'])
    summary.columns = ['_'.join(c) for c in summary.columns]
    print('\n📋 جدول خلاصه نهایی:')
    print(summary.to_string())

    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=['MAE (mean ± std)', 'RMSE (mean ± std)'])
    for col_idx, metric in enumerate(['mae', 'rmse'], start=1):
        grp = results_df.groupby('pred_len')[metric]
        x   = grp.mean().index.astype(str).tolist()
        y   = grp.mean().values
        err = grp.std().values
        fig.add_trace(go.Bar(
            x=x, y=y,
            error_y=dict(type='data', array=err, visible=True),
            name=metric.upper(), showlegend=False),
            row=1, col=col_idx)
        fig.update_xaxes(title_text='Prediction Horizon (days)', row=1, col=col_idx)
        fig.update_yaxes(title_text=metric.upper() + ' (°C)', row=1, col=col_idx)
    fig.update_layout(
        title='Autoformer+FAN — MAE & RMSE across Horizons',
        template='plotly_white', font=dict(size=14), height=450)
    fig.write_html('autoformer_fan_metrics_bar.html')
    print('✅ نمودار در autoformer_fan_metrics_bar.html ذخیره شد.')


Device: cuda

############################################################
#  افق پیش‌بینی: 1 روز
############################################################
کرنل پویا برای pred_len=1 → kernel_size=3

seed=42 | pred_len=1 | kernel=3
Total params: 301,729
  Epoch  1 | Train MSE: 4749.6305 | Val MAE: 11.2792 | Val sMAPE: 163.2065
  ✅ XAI snapshot آپدیت شد — Best Epoch=1 | Val MAE=11.2792
  ✅ XAI snapshot آپدیت شد — Best Epoch=2 | Val MAE=3.6005
  ✅ XAI snapshot آپدیت شد — Best Epoch=4 | Val MAE=2.7061
  ✅ XAI snapshot آپدیت شد — Best Epoch=5 | Val MAE=2.5614
  Epoch  6 | Train MSE: 208.1603 | Val MAE: 2.3819 | Val sMAPE: 46.3573
  ✅ XAI snapshot آپدیت شد — Best Epoch=6 | Val MAE=2.3819
  ✅ XAI snapshot آپدیت شد — Best Epoch=7 | Val MAE=2.3499
  ✅ XAI snapshot آپدیت شد — Best Epoch=8 | Val MAE=2.2182
  Epoch 11 | Train MSE: 101.8075 | Val MAE: 2.1838 | Val sMAPE: 42.7002
  ✅ XAI snapshot آپدیت شد — Best Epoch=11 | Val MAE=2.1838
  ✅ XAI snapshot آپدیت شد — Best Epoch=14 | Val MAE=2.1031
